In [127]:
import pandas as pd

py_df = pd.read_csv('data/py_repos.csv', usecols=['rank', 'username/repo_name'])
all_safetensor_keys = set(f'{row["rank"]}.{row["username/repo_name"]}' for _, row in py_df.iterrows())
all_safetensor_keys

{'100.chubin/wttr.in',
 '11.josephmisiti/awesome-machine-learning',
 '12.public-apis/public-apis',
 '13.donnemartin/system-design-primer',
 '13.huggingface/transformers',
 '15.psf/requests',
 '16.521xueweihan/HelloGitHub',
 '18.scrapy/scrapy',
 '19.soimort/you-get',
 '20.minimaxir/big-list-of-naughty-strings',
 '21.ageitgey/face_recognition',
 '22.apache/superset',
 '23.TheAlgorithms/Python',
 '24.deepfakes/faceswap',
 '25.3b1b/manim',
 '25.jackfrued/Python-100-Days',
 '26.tiangolo/fastapi',
 '26.vinta/awesome-python',
 '27.ytdl-org/youtube-dl',
 '28.fighting41love/funNLP',
 '29.0voice/interview_internal_reference',
 '30.localstack/localstack',
 '31.isocpp/CppCoreGuidelines',
 '32.apachecn/AiLearning',
 '33.pandas-dev/pandas',
 '34.floodsung/Deep-Learning-Papers-Reading-Roadmap',
 '35.testerSunshine/12306',
 '36.faif/python-patterns',
 '37.google-research/bert',
 '38.getsentry/sentry',
 '39.CorentinJ/Real-Time-Voice-Cloning',
 '40.swisskyrepo/PayloadsAllTheThings',
 '41.willmcgugan/ric

In [150]:
from safetensors.torch import load_file
import polars as pl
from pathlib import Path
import torch


schema = [('expert', pl.Int8)]
for _, row in py_df.iterrows():
    repo = row["username/repo_name"]
    schema.append((f'{repo}.w1', pl.Float16))
    schema.append((f'{repo}.w3', pl.Float16))


layer_to_df: dict[int, pl.DataFrame] = {}

for layer in range(32):

    df = pl.DataFrame(schema=schema)
    layer_path = Path(f'output/py/layer={layer:02d}')

    for expert in range(8):
        repos_to_activations = {}
        expert_path = layer_path / f'expert={expert}'

        w1_path = expert_path / 'w=1.safetensors'
        w3_path = expert_path / 'w=3.safetensors'

        w1 = load_file(w1_path)
        w3 = load_file(w3_path)

        assert set(w1.keys()) == all_safetensor_keys, f"Repo missing in w=1 for expert {expert}"
        assert set(w3.keys()) == all_safetensor_keys, f"Repo missing in w=3 for expert {expert}"

        for _, row in py_df.iterrows():
            key = f'{row["rank"]}.{row["username/repo_name"]}'

            w1_activation = w1[key].cpu().to(torch.float16).numpy()
            assert w1_activation.shape == (14336,), f"Unexpected shape for {key} in w=1: {w1_activation.shape}"
            repos_to_activations[f'{row["username/repo_name"]}.w1'] = w1_activation


            w3_activation = w3[key].cpu().to(torch.float16).numpy()
            assert w3_activation.shape == (14336,), f"Unexpected shape for {key} in w=3: {w3_activation.shape}"
            repos_to_activations[f'{row["username/repo_name"]}.w3'] = w3_activation
        
        curr_expert_df = pl.DataFrame(repos_to_activations)
        curr_expert_df = curr_expert_df.with_columns(pl.lit(expert, dtype=pl.Int8).alias("expert"))
        curr_expert_df = curr_expert_df.select(pl.col(df.columns))


        df = pl.concat([df, curr_expert_df], how="vertical")
    
    layer_to_df[layer] = df

In [ ]:
class A:
    pass

dir(A)

x = dir(A)


27

In [154]:
df = layer_to_df[16]

df_top_50_indices = df.drop('expert').select(
    pl.all().arg_sort(descending=True).head(100)
)

df_top_50_indices // 14336

public-apis/public-apis.w1,public-apis/public-apis.w3,donnemartin/system-design-primer.w1,donnemartin/system-design-primer.w3,TheAlgorithms/Python.w1,TheAlgorithms/Python.w3,jackfrued/Python-100-Days.w1,jackfrued/Python-100-Days.w3,vinta/awesome-python.w1,vinta/awesome-python.w3,ytdl-org/youtube-dl.w1,ytdl-org/youtube-dl.w3,tensorflow/models.w1,tensorflow/models.w3,nvbn/thefuck.w1,nvbn/thefuck.w3,django/django.w1,django/django.w3,pallets/flask.w1,pallets/flask.w3,keras-team/keras.w1,keras-team/keras.w3,httpie/httpie.w1,httpie/httpie.w3,scikit-learn/scikit-learn.w1,scikit-learn/scikit-learn.w3,ansible/ansible.w1,ansible/ansible.w3,shadowsocks/shadowsocks.w1,shadowsocks/shadowsocks.w3,python/cpython.w1,python/cpython.w3,home-assistant/core.w1,home-assistant/core.w3,josephmisiti/awesome-machine-learning.w1,josephmisiti/awesome-machine-learning.w3,huggingface/transformers.w1,…,magenta/magenta.w3,locustio/locust.w1,locustio/locust.w3,pytorch/examples.w1,pytorch/examples.w3,jumpserver/jumpserver.w1,jumpserver/jumpserver.w3,luong-komorebi/Awesome-Linux-Software.w1,luong-komorebi/Awesome-Linux-Software.w3,PaddlePaddle/Paddle.w1,PaddlePaddle/Paddle.w3,reddit-archive/reddit.w1,reddit-archive/reddit.w3,charlax/professional-programming.w1,charlax/professional-programming.w3,instillai/TensorFlow-Course.w1,instillai/TensorFlow-Course.w3,python-poetry/poetry.w1,python-poetry/poetry.w3,open-mmlab/mmdetection.w1,open-mmlab/mmdetection.w3,toml-lang/toml.w1,toml-lang/toml.w3,junyanz/pytorch-CycleGAN-and-pix2pix.w1,junyanz/pytorch-CycleGAN-and-pix2pix.w3,bokeh/bokeh.w1,bokeh/bokeh.w3,sanic-org/sanic.w1,sanic-org/sanic.w3,binux/pyspider.w1,binux/pyspider.w3,nginx-proxy/nginx-proxy.w1,nginx-proxy/nginx-proxy.w3,cookiecutter/cookiecutter.w1,cookiecutter/cookiecutter.w3,chubin/wttr.in.w1,chubin/wttr.in.w3
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,…,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
2,2,1,0,0,0,3,0,0,0,1,0,1,0,4,2,3,0,0,7,0,0,0,2,7,0,0,1,2,2,0,7,0,7,0,0,0,…,2,0,0,1,7,0,2,0,1,0,2,1,0,0,1,1,0,0,0,0,0,0,2,1,0,0,7,0,0,0,2,7,7,1,2,0,0
0,0,0,5,2,7,7,6,2,2,0,2,0,0,1,0,1,2,2,2,1,2,0,7,0,5,0,1,0,7,0,0,0,0,1,7,2,…,7,7,7,2,6,0,7,1,7,2,1,2,1,1,0,6,0,1,2,0,0,7,7,7,0,7,0,0,7,1,7,4,4,2,7,3,1
1,1,1,5,7,0,0,0,0,7,7,0,7,0,2,0,0,0,2,0,0,0,0,0,0,7,6,0,2,0,0,1,1,5,7,0,7,…,1,0,2,1,7,2,5,1,2,6,1,0,0,1,7,0,1,0,7,1,2,3,0,4,1,1,2,7,5,1,0,1,2,0,0,0,7
0,5,2,2,0,0,1,0,1,1,0,0,0,0,0,0,7,0,1,5,0,0,2,1,0,2,2,1,1,5,0,0,2,2,1,0,2,…,4,2,1,1,1,7,2,2,0,0,1,0,2,7,0,0,1,2,5,6,0,2,5,6,5,5,5,1,2,7,0,0,5,7,5,4,5
0,6,1,1,3,6,2,0,7,0,2,0,1,1,7,2,0,1,3,1,0,2,1,0,0,0,0,0,0,1,2,0,3,0,2,0,1,…,0,1,1,0,7,0,0,7,1,3,0,2,0,2,1,2,0,0,4,0,0,1,4,0,7,2,0,5,4,6,2,5,1,1,0,1,2
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,0,1,1,1,0,0,1,0,6,7,2,2,5,1,2,0,5,2,2,2,7,0,5,0,0,2,2,0,5,5,4,2,2,7,0,5,…,1,1,7,7,6,0,2,2,1,3,1,2,6,2,0,6,6,7,0,0,1,0,0,1,5,5,5,1,2,0,0,1,1,4,5,7,2
6,0,2,4,0,2,3,6,1,7,0,2,3,6,5,5,0,6,5,6,6,1,2,3,0,0,6,5,0,6,5,0,2,5,3,1,0,…,1,3,0,0,6,1,3,6,6,6,3,0,1,0,1,7,1,0,2,1,1,3,1,3,5,6,3,3,5,4,1,2,3,6,1,7,4
7,1,3,4,3,1,0,0,3,6,0,2,7,0,0,0,2,6,0,7,0,3,2,2,2,1,0,5,0,7,5,3,3,7,5,0,1,…,6,4,7,0,6,0,4,0,1,6,7,2,1,4,1,0,3,4,3,2,2,0,2,0,2,0,0,7,0,6,2,2,3,7,1,0,7
